In [33]:
# Read every csv file located in "/raw_sentiment_outputs" folder

import os
import pandas as pd
import glob

def read_csv_files(directory):
    # Create a list to hold DataFrames
    dataframes = []
    # Use glob to find all csv files in the specified directory
    csv_files = glob.glob(os.path.join(directory, "*.csv"))
    # Loop through each file and read it into a DataFrame
    for file in csv_files:
        df = pd.read_csv(file)
        dataframes.append(df)
    return dataframes

# Specify the directory containing the CSV files
directory = "raw_sentiment_outputs/"
# Read the CSV files
dataframes = read_csv_files(directory)

In [34]:
dataframes[0]

,date,fulltext_clean,section_name,sentiment_10_Year_Treasury
0,2004-01-01,"for the ex-buccaneer, a pillage-free playlist ...",Technology,neutral
1,2004-01-01,year's big rally helps investors regain ground...,Business Day,neutral
2,2004-01-01,world business briefing | asia: india: economi...,Business Day,positive
3,2004-01-01,"flight sent back on terror fear, u.s. official...",U.S.,neutral
4,2004-01-01,world business briefing | europe: britain: clo...,Business Day,neutral
...,...,...,...,...
70600,2024-12-30,"in the presidents’ club, carter was the odd ma...",U.S.,neutral
70601,2024-12-30,"jimmy carter was right about materialism but, ...",Your Money,neutral
70602,2024-12-30,social media companies face global tug-of-war ...,Technology,neutral
70603,2024-12-30,from bird strike to crash: the mystery of the ...,World,neutral


In [ ]:
# For every dataframe we modify the columns starting with "sentiment" so that positive sentiment is 1, negative sentiment is -1, and neutral sentiment is 0

def modify_sentiment_columns(dataframes):
    for df in dataframes:
        # Get all columns that start with 'sentiment'
        sentiment_columns = [col for col in df.columns if col.startswith('sentiment')]
        print(sentiment_columns)
        # Modify the sentiment columns
        for col in sentiment_columns:
            df[col] = df[col].replace({'positive': 1, 'negative': -1, 'neutral': 0})
    return dataframes
# Modify the sentiment columns in all dataframes
dataframes = modify_sentiment_columns(dataframes)

['sentiment_10_Year_Treasury']
['sentiment_Apple']
['sentiment_Coca_Cola']
['sentiment_Disney']
['sentiment_Dow_Jones']
['sentiment_EUR-USD']
['sentiment_Exxon_Mobil']
['sentiment_Federal_Funds_Rate']
['sentiment_Gold']
['sentiment_Google']
['sentiment_Intel']
['sentiment_Johnson_&_Johnson']
['sentiment_JP_Morgan']
['sentiment_Microsoft']
['sentiment_NASDAQ_Composite']
['sentiment_Oil']
['sentiment_S&P_500']
['sentiment_USD-JPY']
['sentiment_Walmart']


In [38]:
dataframes[18]

,date,fulltext_clean,section_name,sentiment_Walmart
0,2004-01-01,"for the ex-buccaneer, a pillage-free playlist ...",Technology,0
1,2004-01-01,year's big rally helps investors regain ground...,Business Day,1
2,2004-01-01,world business briefing | asia: india: economi...,Business Day,1
3,2004-01-01,"flight sent back on terror fear, u.s. official...",U.S.,0
4,2004-01-01,world business briefing | europe: britain: clo...,Business Day,0
...,...,...,...,...
70600,2024-12-30,"in the presidents’ club, carter was the odd ma...",U.S.,0
70601,2024-12-30,"jimmy carter was right about materialism but, ...",Your Money,0
70602,2024-12-30,social media companies face global tug-of-war ...,Technology,0
70603,2024-12-30,from bird strike to crash: the mystery of the ...,World,0


In [39]:
from collections import Counter

processed_dataframes = []

for df in dataframes:
    # Ortalama skor
    sentiment_columns = [col for col in df.columns if col.startswith('sentiment')]
    sentiment_column = sentiment_columns[0]

    df[sentiment_column] = pd.to_numeric(df[sentiment_column], errors='coerce')
    df = df.dropna(subset=[sentiment_column])
    # 2️⃣ Ortalama skor
    mean_scores = df.groupby("date")[sentiment_column].mean().reset_index()
    mean_scores.rename(columns={sentiment_column: "mean_score"}, inplace=True)

    # 3️⃣ Majority Vote (mod)
    def majority_vote(group):
        return Counter(group).most_common(1)[0][0]

    majority_scores = df.groupby("date")[sentiment_column].agg(majority_vote).reset_index()
    majority_scores.rename(columns={sentiment_column: "majority_vote_score"}, inplace=True)

    # 4️⃣ Medyan
    median_scores = df.groupby("date")[sentiment_column].median().reset_index()
    median_scores.rename(columns={sentiment_column: "median_score"}, inplace=True)

    # 5️⃣ Ortalama skoru sınıflandır (-1, 0, 1)
    def classify_mean(x, threshold=0.3):
        if x > threshold:
            return 1
        elif x < -threshold:
            return -1
        else:
            return 0

    mean_scores["mean_class"] = mean_scores["mean_score"].apply(classify_mean)

    # 6️⃣ Birleştir ve Kaydet
    df = mean_scores.merge(majority_scores, on="date").merge(median_scores, on="date")
    processed_dataframes.append(df)
    #final_df.to_csv("daily_finbert_aggregated_with_class.csv", index=False)


In [40]:
processed_dataframes[0]

,date,mean_score,mean_class,majority_vote_score,median_score
0,2004-01-01,0.000000,0,0.0,0.0
1,2004-01-02,-0.666667,-1,-1.0,-1.0
2,2004-01-05,-0.212121,0,0.0,0.0
3,2004-01-06,-0.222222,0,0.0,0.0
4,2004-01-07,-0.105263,0,0.0,0.0
...,...,...,...,...,...
5472,2024-12-24,-0.142857,0,0.0,0.0
5473,2024-12-25,0.000000,0,0.0,0.0
5474,2024-12-26,-0.285714,0,0.0,0.0
5475,2024-12-27,-0.142857,0,0.0,0.0


In [42]:
# Save them to csv
df_names = [ '10_Year_Treasury','Apple','Coca_Cola','Disney','Dow_Jones',
            'EUR-USD','Exxon_Mobil','Federal_Funds_Rate','Gold','Google',
            'Intel','Johnson_&_Johnson','JP_Morgan','Microsoft','NASDAQ_Composite',
            'Oil','S&P_500','USD-JPY','Walmart']

for i, df in enumerate(processed_dataframes):
    df.to_csv(f"processed_outputs/{df_names[i]}_sentiment.csv", index=False)